<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Backend Module 2 (b): FastAPI & CRUD

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Learn

In this notebook, you will:

1. Run a **real FastAPI app right here in Colab** — no server, no port, nothing to install
2. Send a request and read the response yourself, the way another program would
3. Build the **four verbs** — create, read, update, delete
4. See why **PATCH** has a trap in it that quietly erases people's data
5. Return **404, 409 and 422** on purpose, and say what each one means
6. Watch **Pydantic refuse bad data** before your code ever runs

> **Nothing to install and no API key.** Normally a backend needs a running server. Here we use
> FastAPI's **TestClient**, which calls your app directly — same code, same responses, no server.
>
> This is the notebook version of what you saw on the projector. Run it, change things, break it.

## 1. How to Use This Notebook

Run every cell top to bottom, in order — later cells use the app the earlier ones built.

**Don't just read it.** Change a value and run it again. Change `age` to `"abc"`. Delete a field.
Break it on purpose and read the error. That is the whole point of having it.

| Part | Sections | What it is |
|---|---|---|
| **A · The basics** | 2–4 | an app, a response, and data going in |
| **B · All four verbs** | 5–8 | create, read, update, delete — properly |
| **C · Saying no** | 9 | 404, 409, 422 and what each really means |

---

In [ ]:
# One install, about 20 seconds. Nothing else is needed - no server, no port.
!pip install -q fastapi "pydantic[email]" httpx
print("ready")

## 2. Your First Endpoint — and No Server

An endpoint is a function with a URL attached. `@app.get("/health")` means
*"when someone asks for /health, run this."*

`TestClient` is what makes this work in Colab: it calls your app **directly**, in memory. Same code
you would deploy — you just skipped the server.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()


@app.get("/health")           # <- the URL
def health():                 # <- the function that answers it
    return {"status": "ok"}   # <- a dict goes out as JSON


client = TestClient(app)      # our stand-in for a browser

response = client.get("/health")
print("status code:", response.status_code)
print("body       :", response.json())

That `200` is the whole idea of a status code: **the number is the message**. `200` means "here you go".

## 3. Path Params and Query Params

Two ways to send information in a URL, and they mean different things.

- **Path** — `/students/1` — *identifies which thing you want*
- **Query** — `/students?branch=CSE` — *modifies the request*

In [ ]:
@app.get("/students/{student_id}")        # path param, in the URL itself
def get_student(student_id: int):         # this int hint IS the validation
    return {"id": student_id, "name": "Ada"}


print(client.get("/students/1").json())

In [ ]:
# Now ask for a student whose id is not a number at all.
bad = client.get("/students/abc")
print("status:", bad.status_code)
print(bad.json())

**422**, and you did not write a single line of checking. The type hint `student_id: int` was the rule.
Read the message — it tells you exactly which part of the request was wrong.

## 4. Sending Data In — POST and Pydantic

A **Pydantic model** is a class that describes the shape of data *and enforces it*.

In [ ]:
from pydantic import BaseModel, EmailStr, Field


class StudentIn(BaseModel):                       # what we ACCEPT
    name: str = Field(min_length=2)
    branch: str = "CSE"
    age: int = Field(ge=16, le=100)               # must be between 16 and 100
    email: EmailStr                               # must actually look like an email
    password: str


class StudentOut(BaseModel):                      # what we RETURN. No password. On purpose.
    id: int
    name: str
    branch: str
    email: EmailStr


students: list[dict] = []
print("models defined")

In [ ]:
from fastapi import status


@app.post("/students", status_code=status.HTTP_201_CREATED, response_model=StudentOut)
def create_student(incoming: StudentIn):
    student = {"id": len(students) + 1, **incoming.model_dump()}
    students.append(student)
    return student            # the password IS in here...


r = client.post("/students", json={
    "name": "Ada", "age": 20, "email": "ada@lpu.in", "password": "secret123"})
print("status:", r.status_code)
print("body  :", r.json())        # ...but it is NOT in here

💡 **The password vanished.** `response_model=StudentOut` filters the response down to the declared
shape. Leaking a password field is a real bug that has ended real jobs — one line prevents it.

Now send something invalid and watch Pydantic refuse it **before your function runs**.

In [ ]:
# age is below the minimum, and the email is not an email
r = client.post("/students", json={
    "name": "X", "age": 9, "email": "not-an-email", "password": "x"})
print("status:", r.status_code)
for problem in r.json()["detail"]:
    print(" -", problem["loc"][-1], ":", problem["msg"])

Three problems, each named precisely. **You declared what you wanted, not how to check it.**

## 5. Reading — GET the collection

In [ ]:
@app.get("/students", response_model=list[StudentOut])
def list_students():
    return students


client.post("/students", json={
    "name": "Raj", "age": 21, "email": "raj@lpu.in", "password": "secret456"})

print(client.get("/students").json())

## 6. PUT — Replace the Whole Record

`PUT` means *"here is the new record"*. Every field is replaced.

In [ ]:
from fastapi import HTTPException


def find(student_id: int) -> dict:
    for s in students:
        if s["id"] == student_id:
            return s
    raise HTTPException(status.HTTP_404_NOT_FOUND, f"No student with id {student_id}")


@app.put("/students/{student_id}", response_model=StudentOut)
def replace_student(student_id: int, incoming: StudentIn):
    student = find(student_id)
    student.update(incoming.model_dump())
    return student


r = client.put("/students/1", json={
    "name": "Ada Lovelace", "branch": "ECE", "age": 20,
    "email": "ada@lpu.in", "password": "secret123"})
print(r.json())

## 7. PATCH — and the Trap

`PATCH` means *"here is the change"*. Only what you send should move.

The trap: if every field is optional, the ones the caller **didn't send** still show up as `None` —
and a careless update writes those `None`s over real data.

In [ ]:
class StudentPatch(BaseModel):        # every field optional
    name: str | None = None
    branch: str | None = None
    age: int | None = None


@app.patch("/students/{student_id}/broken")     # no response_model, so we see the damage
def broken_patch(student_id: int, changes: StudentPatch):
    student = find(student_id)
    student.update(changes.model_dump())        # <- NO exclude_unset. The bug.
    return student


# The caller only wants to change the branch.
print(client.patch("/students/2/broken", json={"branch": "MECH"}).json())

🚨 **Look at `name` and `age`.** The caller never mentioned them, and they just became `None`. Raj lost his name.

(Notice this endpoint has no `response_model` — that is only so you can *see* the wreckage. With one
declared, FastAPI would refuse to send a `null` name at all and you would get a 500 instead: the bug
would still be there, just harder to spot.)

One argument fixes it.

In [ ]:
@app.patch("/students/{student_id}", response_model=StudentOut)
def update_student(student_id: int, changes: StudentPatch):
    student = find(student_id)
    student.update(changes.model_dump(exclude_unset=True))   # <- the fix
    return student


students[1].update({"name": "Raj", "branch": "ECE", "age": 21})   # put Raj back
print(client.patch("/students/2", json={"branch": "MECH"}).json())

💡 **`exclude_unset=True` is the whole lesson.** *"`None` because they sent null"* and *"`None`
because they said nothing"* are different facts, and this is how you tell them apart.

## 8. DELETE — and the 204 That Confuses Everyone

In [ ]:
@app.delete("/students/{student_id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_student(student_id: int):
    students.remove(find(student_id))
    return None                   # no body. Not an empty one - none at all.


r = client.delete("/students/2")
print("status :", r.status_code)
print("body   :", repr(r.content))       # genuinely empty
print("again  :", client.delete("/students/2").status_code)   # now it is gone

**204** means *"done — and there is deliberately nothing to show you"*. The status code IS the message.

## 9. Three Ways to Fail, and They Are Not the Same

This is the part that separates an API somebody can actually use from one they cannot.

In [ ]:
@app.post("/students/careful", status_code=201, response_model=StudentOut)
def create_careful(incoming: StudentIn):
    if any(s["email"] == incoming.email for s in students):
        raise HTTPException(status.HTTP_409_CONFLICT, f"{incoming.email} is already registered")
    student = {"id": len(students) + 1, **incoming.model_dump()}
    students.append(student)
    return student


body = {"name": "Meera", "age": 22, "email": "meera@lpu.in", "password": "secret789"}
print("first time :", client.post("/students/careful", json=body).status_code)
print("second time:", client.post("/students/careful", json=body).status_code)

| Code | Means | Who raised it |
|---|---|---|
| **422** | the request is **malformed** | Pydantic, before your function ran |
| **404** | the request is fine, **the thing doesn't exist** | you |
| **409** | the request is fine, **the thing already exists** | you |

> **404 is "it isn't there". 409 is "it's already there".** Those are the two halves of *"the world
> isn't how your request assumed it is"*.

---

## 10. Exercises

Fill in the `___`. The app from above is still loaded.

### Q1. An endpoint that returns your own name

**Hint:** copy the `/health` shape. A dict goes out as JSON.

In [ ]:
@app.get("/___")
def me():
    return {"name": "___"}


print(client.get("/___").json())

### Q2. Ask for a student who does not exist, and print the status code

**Hint:** `find()` already raises the right error. You only need to call the endpoint.

In [ ]:
r = client.get("/students/___")
print(r.status_code)

### Q3. Create a student whose age is 200

**Hint:** the rule was `Field(ge=16, le=100)`. What number should come back?

In [ ]:
r = client.post("/students", json={
    "name": "Old", "age": ___, "email": "old@lpu.in", "password": "secret123"})
print(r.status_code, r.json()["detail"][0]["msg"])

### Q4. PATCH student 1 so only the branch changes, and prove the name survived

**Hint:** use the endpoint that has `exclude_unset=True`, not the broken one.

In [ ]:
print(client.patch("/students/1", json={"___": "CIVIL"}).json())

---

## Key Takeaways

1. **The number is the message.** 200 here you go · 201 created · 204 done, nothing to show · 404 no such thing · 409 already exists · 422 wrong shape.
2. **You declare rules, you don't write checks.** A type hint and a `Field()` did all the validation in this notebook.
3. **`response_model` filters the response** — the password never left, even though the function returned it.
4. **PUT replaces, PATCH changes** — and `exclude_unset=True` is what stops PATCH erasing data nobody asked it to touch.
5. **A path names a thing, a method says what to do to it.** `DELETE /students/1`, never `GET /deleteStudent`.

> Next: the data in this notebook still disappears when the runtime restarts. The next notebook gives it a real database.